# Baseline Data-Quality and Anomaly-Detection Pipeline

This notebook implements the reproducible baseline pipeline for the accompanying study. It creates controlled anomaly experiments, deterministic rule evidence, Isolation Forest and Local Outlier Factor outputs, and validated artifacts consumed by the governed decision engine.

## Execution order

Run this notebook first, then run `02_Hybrid_Decision_Engine.ipynb`.

## Core safeguards

- prevalence-preserving experiment manifests
- rule evaluation against each corrupted experiment
- held-out clean detector-training partitions
- one-to-one record and experiment identifiers
- executable assertions for prevalence, class coverage, and artifact completeness
- reproducible, versioned outputs


## 1. Install dependencies
Colab already includes many scientific packages, but this cell installs the versions needed for Excel and Parquet support.

In [ ]:
!pip -q install openpyxl pyarrow scikit-learn joblib

## 2. Configure Repository Storage

The default configuration uses the repository root and therefore works in a local clone, Codespace, or Jupyter environment. In Google Colab, set `USE_GOOGLE_DRIVE = True` and optionally change `GOOGLE_DRIVE_PROJECT_ROOT`.


In [ ]:
from pathlib import Path
import sys

# ----------------------------------------------------------------
# Google Drive project root
# ----------------------------------------------------------------
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Paper2_ZTLF")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE=True requires a Google Colab runtime."
        ) from exc
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = GOOGLE_DRIVE_PROJECT_ROOT
else:
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir

RAW_DIR        = PROJECT_ROOT / "data" / "raw"
BRONZE_DIR     = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR     = PROJECT_ROOT / "data" / "silver"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments"
RESULTS_DIR    = PROJECT_ROOT / "results"
MODEL_DIR      = PROJECT_ROOT / "models"
FIGURES_DIR    = PROJECT_ROOT / "figures"
TABLES_DIR     = PROJECT_ROOT / "tables"

MANIFEST_DIR      = RESULTS_DIR / "sample_manifests"
RULE_BY_EXP_DIR   = RESULTS_DIR / "rule_results_by_experiment"
AI_RESULT_DIR     = RESULTS_DIR / "ai_results_by_experiment"

for directory in [RAW_DIR, BRONZE_DIR, SILVER_DIR, EXPERIMENT_DIR, RESULTS_DIR,
                  MODEL_DIR, FIGURES_DIR, TABLES_DIR,
                  MANIFEST_DIR, RULE_BY_EXP_DIR, AI_RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------
# Global run configuration
# ----------------------------------------------------------------
RANDOM_SEED = 42

# Records evaluated per experiment. This is the primary memory control.
SAMPLE_RECORDS_PER_EXPERIMENT = 10_000

# Fraction of clean reference data reserved for fitting the detectors.
TRAIN_FRACTION = 0.40
MAX_TRAIN_ROWS = {"finance": 40_000, "healthcare": 40_000, "retail": 50_000}

# Integrity thresholds.
PREVALENCE_TOLERANCE  = 0.005
MIN_NEGATIVE_RECORDS  = 100
MIN_POSITIVE_RECORDS  = 30

# Set False only to resume a genuinely interrupted run, and only when you are
# certain the existing files came from THIS version of the notebook.
PURGE_STALE_ARTIFACTS = True


# ----------------------------------------------------------------
# Memory reporting helper
# ----------------------------------------------------------------
import gc
import psutil
import os

def memory_report(label=""):
    """Print current process RSS. Use liberally; it costs nothing."""
    gc.collect()
    rss_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    available_gb = psutil.virtual_memory().available / (1024 ** 3)
    print(f"[MEM] {label:<38} used={rss_gb:5.2f} GB   available={available_gb:5.2f} GB")
    return rss_gb

print("Project root :", PROJECT_ROOT)
print("Sample size  :", f"{SAMPLE_RECORDS_PER_EXPERIMENT:,} records per experiment")
print("Purge stale  :", PURGE_STALE_ARTIFACTS)
memory_report("after setup")


In [ ]:
print("Raw directory:", RAW_DIR)

if not RAW_DIR.exists():
    print("Directory does not exist. Check the Google Drive path.")
else:
    files = [p for p in sorted(RAW_DIR.iterdir()) if p.is_file()]
    print(f"\nFiles found: {len(files)}")
    for file_path in files:
        print(f"  {file_path.name:<30} {file_path.stat().st_size / (1024*1024):8.2f} MB")


## 3. Upload or copy source files
Required files:
- `bank-full.csv` (preferred) or `bank.csv`
- `diabetic_data.csv`
- `online_retail_II.xlsx`
- Optional: `IDS_mapping.csv`

When using Drive, copy the files into `Paper2_ZTLF/data/raw/`. For a one-time upload into the Colab session, set `USE_GOOGLE_DRIVE = False` and run the upload cell below.

In [ ]:
# Optional one-time upload. Skip when files already exist in RAW_DIR.
# from google.colab import files
# uploaded = files.upload()
# for filename, content in uploaded.items():
#   (RAW_DIR / filename).write_bytes(content)

## 4. Imports and reproducibility configuration

In [ ]:
import json
import re
import time
import hashlib
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

def calculate_false_positive_rate(y_true, y_pred):
    """
    Calculate the false-positive rate:
    FPR = FP / (FP + TN)
    """
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    denominator = fp + tn
    return fp / denominator if denominator > 0 else 0.0

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
pd.set_option('display.max_columns', 100)

## 5. Data ingestion and Bronze-layer artifact creation
Source datasets are loaded, normalized, enriched with audit metadata and deterministic record identifiers, and persisted as reproducible Parquet artifacts for downstream experiments.

In [ ]:
def normalize_column_name(name: str) -> str:
    normalized = re.sub(r'[^a-zA-Z0-9]+', '_', str(name).strip().lower())
    return re.sub(r'_+', '_', normalized).strip('_')


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [normalize_column_name(c) for c in result.columns]
    return result


def add_audit_columns(df: pd.DataFrame, dataset: str, source_file: str) -> pd.DataFrame:
    result = df.copy()
    result['_source_row_number'] = np.arange(len(result), dtype='int64')
    result['_dataset'] = dataset
    result['_source_file'] = source_file
    result['_ingestion_timestamp_utc'] = datetime.now(timezone.utc).isoformat()
    result['_record_id'] = [f'{dataset}_{i:09d}' for i in range(len(result))]
    return result


def locate_finance_file() -> Path:
    preferred = RAW_DIR / 'bank-full.csv'
    fallback = RAW_DIR / 'bank.csv'
    if preferred.exists():
        return preferred
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Place bank-full.csv or bank.csv in RAW_DIR.')

finance_path = locate_finance_file()
health_path = RAW_DIR / 'diabetic_data.csv'
retail_path = RAW_DIR / 'online_retail_II.xlsx'

for required in [health_path, retail_path]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required file: {required}')

finance_df = pd.read_csv(finance_path, sep=';')
health_df = pd.read_csv(health_path, na_values=['?'])

retail_sheets = pd.read_excel(retail_path, sheet_name=None)
retail_df = pd.concat(retail_sheets.values(), ignore_index=True)

# Stabilize known mixed-type identifier and text columns before Parquet writing.
retail_string_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Customer ID",
    "Country",
]

for column in retail_string_columns:
    if column in retail_df.columns:
        retail_df[column] = retail_df[column].astype("string")

bronze = {
    'finance': add_audit_columns(normalize_columns(finance_df), 'finance', finance_path.name),
    'healthcare': add_audit_columns(normalize_columns(health_df), 'healthcare', health_path.name),
    'retail': add_audit_columns(normalize_columns(retail_df), 'retail', retail_path.name),
}

for dataset, df in bronze.items():
    output = BRONZE_DIR / f'bronze_{dataset}.parquet'
    df.to_parquet(output, index=False)
    print(dataset, df.shape, '->', output)

## 6. Profiling
Produces dataset-level metadata, missing-value counts, type information, distinct counts, and descriptive statistics.

In [ ]:
def profile_dataset(df: pd.DataFrame, dataset_name: str):
    missing = df.isna().sum().rename('missing_count').to_frame()
    missing['missing_rate'] = missing['missing_count'] / max(len(df), 1)
    missing['dtype'] = df.dtypes.astype(str)
    missing['distinct_count'] = [df[c].nunique(dropna=True) for c in df.columns]
    missing['dataset'] = dataset_name
    missing['column_name'] = missing.index
    missing = missing.reset_index(drop=True)

    numeric_summary = df.select_dtypes(include=np.number).describe().T.reset_index()
    numeric_summary = numeric_summary.rename(columns={'index': 'column_name'})
    numeric_summary['dataset'] = dataset_name
    return missing, numeric_summary

metadata_rows, missing_frames, summary_frames = [], [], []
for dataset, df in bronze.items():
    missing, summary = profile_dataset(df, dataset)
    missing_frames.append(missing)
    summary_frames.append(summary)
    metadata_rows.append({
        'dataset': dataset,
        'rows': len(df),
        'columns': len(df.columns),
        'profiled_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    })

metadata_df = pd.DataFrame(metadata_rows)
missing_profile_df = pd.concat(missing_frames, ignore_index=True)
numeric_summary_df = pd.concat(summary_frames, ignore_index=True)

metadata_df.to_csv(RESULTS_DIR / 'dataset_metadata.csv', index=False)
missing_profile_df.to_csv(RESULTS_DIR / 'missing_value_profile.csv', index=False)
numeric_summary_df.to_csv(RESULTS_DIR / 'numeric_summary.csv', index=False)

display(metadata_df)

## 7. Deterministic rule configuration
These rules define the rule-based data-quality baseline. Rule violations are retained as auditable, record-level artifacts for later comparison with AI-only and hybrid methods.


In [ ]:
RULES = pd.DataFrame([
    ['FIN_A1_001','finance','Missing age','MISSING_VALUE','age','HIGH',0.25,'QUARANTINE'],
    ['FIN_A3_001','finance','Invalid age','DOMAIN_CONSTRAINT','age','HIGH',0.25,'QUARANTINE'],
    ['FIN_A3_002','finance','Invalid balance','DOMAIN_CONSTRAINT','balance','MEDIUM',0.15,'REVIEW'],
    ['FIN_A3_003','finance','Invalid campaign count','DOMAIN_CONSTRAINT','campaign','MEDIUM',0.15,'REVIEW'],
    # PATCH D: duplicate rules were implemented in the engine but never registered here.
    ['FIN_DUP_001','finance','Duplicate business key','DUPLICATE',
     'age|job|marital|education|balance|day|month|campaign','HIGH',0.25,'QUARANTINE'],

    ['HLT_A1_001','healthcare','Missing encounter identifier','MISSING_VALUE','encounter_id','CRITICAL',0.40,'QUARANTINE'],
    ['HLT_A1_002','healthcare','Missing patient identifier','MISSING_VALUE','patient_nbr','CRITICAL',0.40,'QUARANTINE'],
    ['HLT_A3_001','healthcare','Invalid hospital stay duration','DOMAIN_CONSTRAINT','time_in_hospital','HIGH',0.25,'REVIEW'],
    ['HLT_A3_002','healthcare','Invalid inpatient visit count','DOMAIN_CONSTRAINT','number_inpatient','MEDIUM',0.15,'REVIEW'],
    ['HLT_DUP_001','healthcare','Duplicate encounter','DUPLICATE','encounter_id','HIGH',0.25,'QUARANTINE'],

    ['RTL_A1_001','retail','Missing invoice','MISSING_VALUE','invoice','CRITICAL',0.40,'QUARANTINE'],
    ['RTL_A1_002','retail','Missing stock code','MISSING_VALUE','stockcode','HIGH',0.25,'QUARANTINE'],
    ['RTL_A3_001','retail','Zero quantity','DOMAIN_CONSTRAINT','quantity','HIGH',0.25,'REVIEW'],
    ['RTL_A3_002','retail','Invalid unit price','DOMAIN_CONSTRAINT','price','HIGH',0.25,'REVIEW'],
    ['RTL_DUP_001','retail','Duplicate invoice line','DUPLICATE',
     'invoice|stockcode|quantity|invoicedate|price|customer_id','HIGH',0.25,'QUARANTINE'],
], columns=['rule_id','dataset','rule_name','rule_type','column_name','severity','trust_penalty','recommendation'])

RULES['rule_version'] = '2.0'
RULES['enabled'] = True
RULES.to_csv(RESULTS_DIR / 'dq_rule_config.csv', index=False)

# PATCH: severity is now config-driven for every rule, including duplicates.
SEVERITY_SCORE = {'LOW': 0.10, 'MEDIUM': 0.40, 'HIGH': 0.70, 'CRITICAL': 1.00}

print(f"Registered rules: {len(RULES)}")
print(RULES.groupby(['dataset', 'rule_type']).size().rename('rules').reset_index().to_string(index=False))
RULES


In [ ]:
# ======================================================================
# PATCH B (part 1 of 2) — rule engine DEFINITION only.
#
# In the original notebook this cell ended with:
#     for dataset, df in bronze.items():
#         results, violations = evaluate_rules(df, dataset)
#
# That evaluated rules against CLEAN pre-injection data. The rule engine
# therefore never saw a single injected anomaly, which is why rule-only
# recall was 0.043 and healthcare rule-only F1/FPR were exactly 0.0.
#
# Execution now happens AFTER anomaly injection, per experiment, in the
# new cell that follows Section 8.
# ======================================================================

DUPLICATE_KEYS = {
    # PATCH D: '_record_id' is unique by construction, so the original
    # finance duplicate rule could never fire. Replaced with a composite
    # business key.
    'finance': ['age', 'job', 'marital', 'education', 'balance',
                'day', 'month', 'campaign'],
    'healthcare': ['encounter_id'],
    'retail': ['invoice', 'stockcode', 'quantity',
               'invoicedate', 'price', 'customer_id'],
}

DUPLICATE_RULE_ID = {
    'finance': 'FIN_DUP_001',
    'healthcare': 'HLT_DUP_001',
    'retail': 'RTL_DUP_001',
}


def evaluate_rules(df: pd.DataFrame, dataset: str,
                   duplicate_record_ids: set | None = None
                   ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate deterministic data-quality rules over one dataframe.

    Returns (record_level_results, violation_rows).
    """
    if dataset == 'finance':
        masks = {
            'FIN_A1_001': df['age'].isna(),
            'FIN_A3_001': df['age'].notna() & ~df['age'].between(18, 100),
            'FIN_A3_002': df['balance'].notna() & (df['balance'] < 0),
            'FIN_A3_003': df['campaign'].notna() & (df['campaign'] < 0),
        }
    elif dataset == 'healthcare':
        masks = {
            'HLT_A1_001': df['encounter_id'].isna(),
            'HLT_A1_002': df['patient_nbr'].isna(),
            'HLT_A3_001': df['time_in_hospital'].notna() & ~df['time_in_hospital'].between(1, 14),
            'HLT_A3_002': df['number_inpatient'].notna() & (df['number_inpatient'] < 0),
        }
    elif dataset == 'retail':
        invoice_missing = df['invoice'].isna() | df['invoice'].astype('string').str.strip().eq('')
        stock_missing = df['stockcode'].isna() | df['stockcode'].astype('string').str.strip().eq('')
        masks = {
            'RTL_A1_001': invoice_missing.fillna(True),
            'RTL_A1_002': stock_missing.fillna(True),
            'RTL_A3_001': df['quantity'].eq(0),
            'RTL_A3_002': df['price'].isna() | (df['price'] <= 0),
        }
    else:
        raise ValueError(dataset)

    duplicate_keys = [c for c in DUPLICATE_KEYS[dataset] if c in df.columns]
    duplicate_rule_id = DUPLICATE_RULE_ID[dataset]

    violation_rows = []
    record = df[['_record_id']].copy()
    record['dataset'] = dataset
    record['failed_rule_count'] = 0
    record['trust_score'] = 1.0
    record['max_severity_score'] = 0.0
    record['failed_rules'] = [[] for _ in range(len(record))]

    rule_lookup = RULES.set_index('rule_id')

    def apply_mask(rule_id, mask):
        mask = mask.fillna(False).astype(bool)
        if not mask.any():
            return
        meta = rule_lookup.loc[rule_id]
        severity_score = SEVERITY_SCORE[meta['severity']]
        record.loc[mask, 'failed_rule_count'] += 1
        record.loc[mask, 'trust_score'] -= float(meta['trust_penalty'])
        record.loc[mask, 'max_severity_score'] = np.maximum(
            record.loc[mask, 'max_severity_score'], severity_score
        )
        record.loc[mask, 'failed_rules'] = record.loc[mask, 'failed_rules'].map(
            lambda x: x + [rule_id]
        )
        failing = df.loc[mask, ['_record_id']].copy()
        failing['dataset'] = dataset
        failing['rule_id'] = rule_id
        failing['rule_name'] = meta['rule_name']
        failing['severity'] = meta['severity']
        failing['severity_score'] = severity_score
        failing['recommendation'] = meta['recommendation']
        violation_rows.append(failing)

    for rule_id, mask in masks.items():
        apply_mask(rule_id, mask)

    # Duplicate evidence. When duplicate_record_ids is supplied it was computed
    # on the FULL experiment before sampling, so a duplicate pair split across
    # the sample boundary is still detected. Computing it here on a sampled
    # frame would silently miss those pairs.
    if duplicate_record_ids is not None:
        apply_mask(
            duplicate_rule_id,
            df["_record_id"].astype(str).isin(duplicate_record_ids),
        )
    elif duplicate_keys:
        apply_mask(duplicate_rule_id, df.duplicated(subset=duplicate_keys, keep=False))

    record['trust_score'] = record['trust_score'].clip(0, 1)
    record['rule_status'] = np.where(record['failed_rule_count'].eq(0), 'PASS', 'FAIL')
    record['rule_risk_score'] = (1.0 - record['trust_score']).clip(0, 1)

    violations = (
        pd.concat(violation_rows, ignore_index=True)
        if violation_rows else pd.DataFrame()
    )
    return record, violations


print("evaluate_rules defined. Rules are executed per experiment after injection.")


## 8. Controlled anomaly injection
This baseline creates reproducible experiments at 1%, 5%, and 10% anomaly rates. It preserves clean reference data and writes explicit ground-truth labels.

Implemented baseline anomaly classes:
- A1: missingness
- A2: duplicates
- A3: domain violations
- A4: statistical outliers
- A5: contextual anomalies
- A6: distribution drift
- A7: cross-attribute inconsistencies

In [ ]:
ANOMALY_RATES = [0.01, 0.05, 0.10]
ANOMALY_TYPES = ['A1','A2','A3','A4','A5','A6','A7']

DATASET_CONFIG = {
    'finance': {
        'record_key': '_record_id', 'missing_column': 'age', 'numeric_column': 'balance',
        'domain_column': 'campaign', 'context_columns': ('age','job'), 'cross_columns': ('housing','loan')
    },
    'healthcare': {
        'record_key': '_record_id', 'missing_column': 'encounter_id', 'numeric_column': 'num_lab_procedures',
        'domain_column': 'time_in_hospital', 'context_columns': ('age','time_in_hospital'), 'cross_columns': ('diabetesmed','change')
    },
    'retail': {
        'record_key': '_record_id', 'missing_column': 'invoice', 'numeric_column': 'quantity',
        'domain_column': 'price', 'context_columns': ('quantity','price'), 'cross_columns': ('quantity','price')
    },
}


def inject_anomaly(df: pd.DataFrame, dataset: str, anomaly_type: str, rate: float, seed: int):
    cfg = DATASET_CONFIG[dataset]
    out = df.copy(deep=True)
    n = max(1, int(round(len(out) * rate)))
    selected = out.sample(n=min(n, len(out)), random_state=seed).index
    original_ids = out.loc[selected, '_record_id'].astype(str).tolist()

    if anomaly_type == 'A1':
        out.loc[selected, cfg['missing_column']] = np.nan
    elif anomaly_type == 'A2':
        duplicates = out.loc[selected].copy()
        # PATCH D: keep provenance so the injected duplicate remains traceable
        # to its source row. Detection itself relies on the composite business
        # key in DUPLICATE_KEYS, not on _record_id.
        duplicates['_original_record_id'] = original_ids
        duplicates['_record_id'] = [f'{rid}_dup_{seed}' for rid in original_ids]
        if '_original_record_id' not in out.columns:
            out['_original_record_id'] = out['_record_id']
        out = pd.concat([out, duplicates], ignore_index=True)
        original_ids = duplicates['_record_id'].astype(str).tolist()
    elif anomaly_type == 'A3':
        col = cfg['domain_column']
        out.loc[selected, col] = -999
    elif anomaly_type == 'A4':
        col = cfg['numeric_column']
        numeric = pd.to_numeric(out[col], errors='coerce')
        scale = numeric.std(skipna=True) or 1.0
        center = numeric.median(skipna=True) or 0.0
        out.loc[selected, col] = center + 12 * scale
    elif anomaly_type == 'A5':
        c1, c2 = cfg['context_columns']
        if dataset == 'finance':
            out.loc[selected, c1] = 18
            out.loc[selected, c2] = 'retired'
        elif dataset == 'healthcare':
            out.loc[selected, c1] = '[0-10)'
            out.loc[selected, c2] = 14
        else:
            out.loc[selected, c1] = 1
            out.loc[selected, c2] = pd.to_numeric(out[c2], errors='coerce').median() * 50
    elif anomaly_type == 'A6':
        col = cfg['numeric_column']
        out.loc[selected, col] = pd.to_numeric(out.loc[selected, col], errors='coerce') * 3 + 100
    elif anomaly_type == 'A7':
        c1, c2 = cfg['cross_columns']
        if dataset == 'finance':
            out.loc[selected, [c1,c2]] = 'yes'
        elif dataset == 'healthcare':
            out.loc[selected, c1] = 'No'
            out.loc[selected, c2] = 'Ch'
        else:
            out.loc[selected, c1] = -abs(pd.to_numeric(out.loc[selected, c1], errors='coerce').fillna(1))
            out.loc[selected, c2] = abs(pd.to_numeric(out.loc[selected, c2], errors='coerce').fillna(1))
    else:
        raise ValueError(anomaly_type)

    experiment_id = f'{dataset}_{anomaly_type}_r{int(rate*100):02d}_s{seed}'
    ground_truth = pd.DataFrame({
        'experiment_id': experiment_id,
        'dataset': dataset,
        'anomaly_type': anomaly_type,
        'anomaly_rate': rate,
        'record_id': original_ids,
        'ground_truth_label': 1,
        'injection_seed': seed,
    })
    out['_experiment_id'] = experiment_id
    return experiment_id, out, ground_truth

experiment_registry = []
ground_truth_frames = []

for d_idx, (dataset, reference_df) in enumerate(bronze.items()):
    for a_idx, anomaly_type in enumerate(ANOMALY_TYPES):
        for r_idx, rate in enumerate(ANOMALY_RATES):
            seed = RANDOM_SEED + d_idx*1000 + a_idx*100 + r_idx
            experiment_id, corrupted, gt = inject_anomaly(reference_df, dataset, anomaly_type, rate, seed)
            corrupted.to_parquet(EXPERIMENT_DIR / f'{experiment_id}.parquet', index=False)
            ground_truth_frames.append(gt)
            experiment_registry.append({
                'experiment_id': experiment_id, 'dataset': dataset,
                'anomaly_type': anomaly_type, 'anomaly_rate': rate,
                'reference_rows': len(reference_df), 'experiment_rows': len(corrupted),
                'injected_records': len(gt), 'seed': seed,
            })

experiment_registry_df = pd.DataFrame(experiment_registry)
anomaly_ground_truth_df = pd.concat(ground_truth_frames, ignore_index=True)
experiment_registry_df.to_csv(RESULTS_DIR / 'experiment_registry.csv', index=False)
anomaly_ground_truth_df.to_parquet(RESULTS_DIR / 'anomaly_ground_truth.parquet', index=False)
display(experiment_registry_df.head())

## 8b. Sampling manifest

One prevalence-preserving sample per experiment, written once to disk. Both the rule engine
and the detectors read this manifest, so their evidence covers exactly the same records and
the downstream join in Notebook 2 is one-to-one.

This is also the primary memory control: every later stage works on 10,000 records per
experiment instead of the full corpus, which for retail is 1.07 million rows.


In [ ]:
# ======================================================================
# Build the sampling manifest.
#
# Each experiment gets a fixed-size sample whose anomaly prevalence matches
# the nominal injection rate, with detector-training records excluded.
#
# Memory: only ['_record_id'] is read from each experiment file.
# ======================================================================

import gc
import time

import numpy as np
import pandas as pd


if PURGE_STALE_ARTIFACTS:
    purged = 0
    for path in MANIFEST_DIR.glob("*_manifest.parquet"):
        path.unlink()
        purged += 1
    print(f"Purged {purged} stale manifest files.")


if "experiment_registry_df" not in globals():
    experiment_registry_df = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")

ground_truth_df = pd.read_parquet(
    RESULTS_DIR / "anomaly_ground_truth.parquet",
    columns=["experiment_id", "record_id"],
)
truth_by_experiment = {
    experiment_id: set(group["record_id"].astype(str))
    for experiment_id, group in ground_truth_df.groupby("experiment_id")
}
del ground_truth_df
gc.collect()


# ----------------------------------------------------------------
# Deterministic detector-training record ids.
#
# Defined here rather than in the model cell so that the manifest does not
# depend on models having been fitted first. The model cell reuses this
# function, so the two are guaranteed to agree.
# ----------------------------------------------------------------
import hashlib

_TRAINING_ID_CACHE = {}


def _stable_bucket(record_id: str) -> int:
    """Deterministic 0-999 bucket for a record id."""
    return int(hashlib.sha256(record_id.encode("utf-8")).hexdigest()[:8], 16) % 1000


def get_training_record_ids(dataset: str) -> set:
    """Return the record ids reserved for fitting the detectors.

    Partitioning is by stable hash, NOT by pandas sampling.

    This matters. The earlier seed-based version used
    ``ids.sample(n=train_n, random_state=RANDOM_SEED)`` while the injection
    routine used ``out.sample(n=n, random_state=seed)`` on a frame of the same
    length. Pandas draws a single permutation per seed, so ``sample(n=120,
    random_state=42)`` is a strict prefix of ``sample(n=4000, random_state=42)``.
    Every experiment whose injection seed equalled RANDOM_SEED therefore had
    ALL of its injected anomalies swallowed by the training partition and then
    excluded from scoring, leaving zero positives.

    A hash partition is independent of every injection seed and is stable
    across runs and row-count changes.
    """
    if dataset in _TRAINING_ID_CACHE:
        return _TRAINING_ID_CACHE[dataset]

    ids = pd.read_parquet(
        BRONZE_DIR / f"bronze_{dataset}.parquet", columns=["_record_id"]
    )["_record_id"].astype(str)

    cutoff = int(TRAIN_FRACTION * 1000)
    training = ids[ids.map(_stable_bucket) < cutoff]

    if len(training) > MAX_TRAIN_ROWS[dataset]:
        training = training.iloc[: MAX_TRAIN_ROWS[dataset]]

    result = set(training)
    _TRAINING_ID_CACHE[dataset] = result

    del ids, training
    gc.collect()
    return result


# ----------------------------------------------------------------
# Build one manifest per experiment
# ----------------------------------------------------------------
manifest_started = time.perf_counter()
manifest_rows = []

for position, experiment in experiment_registry_df.iterrows():
    experiment_id = experiment["experiment_id"]
    dataset       = experiment["dataset"]
    nominal_rate  = float(experiment["anomaly_rate"])

    manifest_path = MANIFEST_DIR / f"{experiment_id}_manifest.parquet"
    if manifest_path.exists():
        continue

    record_ids = pd.read_parquet(
        EXPERIMENT_DIR / f"{experiment_id}.parquet", columns=["_record_id"]
    )["_record_id"].astype(str)

    training_ids = get_training_record_ids(dataset)
    eligible = record_ids[~record_ids.isin(training_ids)]

    injected_ids = truth_by_experiment.get(experiment_id, set())
    is_injected = eligible.isin(injected_ids)

    positives = eligible[is_injected]
    negatives = eligible[~is_injected]

    # ------------------------------------------------------------
    # Sample to the NOMINAL injection rate, not the pool rate.
    #
    # The pool rate is not the nominal rate for every class. A2 appends
    # duplicate rows with fresh record ids, so those rows are never in the
    # hash-based training partition while ~40% of the originals are. The
    # pool therefore over-represents anomalies (about 7.7% when nominal is
    # 5%). Targeting the nominal rate directly keeps every experiment
    # comparable and makes the assertion below meaningful.
    # ------------------------------------------------------------
    limit = min(SAMPLE_RECORDS_PER_EXPERIMENT, len(eligible))

    positive_n = min(len(positives), max(1, round(limit * nominal_rate)))
    negative_n = min(len(negatives), limit - positive_n)

    # If negatives are the binding constraint, shrink positives to hold the ratio.
    if negative_n < limit - positive_n:
        positive_n = min(
            positive_n,
            max(1, round(negative_n * nominal_rate / max(1.0 - nominal_rate, 1e-9))),
        )

    positives = positives.sample(n=positive_n, random_state=RANDOM_SEED)
    negatives = negatives.sample(n=negative_n, random_state=RANDOM_SEED)

    sample_ids = pd.concat([positives, negatives], ignore_index=True)
    labels     = sample_ids.isin(injected_ids).astype(int)

    n_positive = int(labels.sum())
    n_negative = int(len(labels) - n_positive)
    realized   = n_positive / max(len(labels), 1)

    # ------------------------------------------------------------
    # Integrity assertions. These are the guards that were missing.
    # ------------------------------------------------------------
    assert n_positive > 0, (
        f"{experiment_id}: no injected anomalies survived the training-partition "
        f"exclusion. Check that the training partition is hash-based and does not "
        f"share a seed with the injection sampler."
    )
    assert abs(realized - nominal_rate) < PREVALENCE_TOLERANCE, (
        f"{experiment_id}: realized prevalence {realized:.4f} vs nominal "
        f"{nominal_rate:.4f} (tolerance {PREVALENCE_TOLERANCE})"
    )
    assert n_negative >= MIN_NEGATIVE_RECORDS, (
        f"{experiment_id}: only {n_negative} negatives. FPR and specificity "
        f"would be undefined — this is the defect that invalidated the earlier run."
    )
    assert n_positive >= MIN_POSITIVE_RECORDS, (
        f"{experiment_id}: only {n_positive} positives."
    )

    pd.DataFrame({
        "experiment_id": experiment_id,
        "dataset": dataset,
        "record_id": sample_ids.values,
        "ground_truth_label": labels.values,
    }).to_parquet(manifest_path, index=False)

    manifest_rows.append({
        "experiment_id": experiment_id,
        "dataset": dataset,
        "anomaly_type": experiment["anomaly_type"],
        "anomaly_rate": nominal_rate,
        "sampled_records": len(sample_ids),
        "positive_records": n_positive,
        "negative_records": n_negative,
        "realized_prevalence": realized,
        "pool_records": len(eligible),
    })

    del record_ids, eligible, positives, negatives, sample_ids, labels
    gc.collect()

    if (position + 1) % 10 == 0:
        memory_report(f"manifest {position + 1}/{len(experiment_registry_df)}")

# Rebuild the summary from disk so a resumed run still produces a complete table.
manifest_summary_df = pd.concat(
    [pd.read_parquet(p, columns=["experiment_id", "dataset", "ground_truth_label"])
       .groupby(["experiment_id", "dataset"], as_index=False)
       .agg(sampled_records=("ground_truth_label", "size"),
            positive_records=("ground_truth_label", "sum"))
     for p in sorted(MANIFEST_DIR.glob("*_manifest.parquet"))],
    ignore_index=True,
)
manifest_summary_df["negative_records"] = (
    manifest_summary_df["sampled_records"] - manifest_summary_df["positive_records"]
)
manifest_summary_df["realized_prevalence"] = (
    manifest_summary_df["positive_records"] / manifest_summary_df["sampled_records"]
)
manifest_summary_df = manifest_summary_df.merge(
    experiment_registry_df[["experiment_id", "anomaly_type", "anomaly_rate"]],
    on="experiment_id", how="left",
)
manifest_summary_df.to_csv(RESULTS_DIR / "sample_manifest_summary.csv", index=False)

assert len(manifest_summary_df) == len(experiment_registry_df), \
    "manifest missing for some experiments"

drift = (manifest_summary_df["realized_prevalence"]
         - manifest_summary_df["anomaly_rate"]).abs()
assert (drift < PREVALENCE_TOLERANCE).all(), \
    f"prevalence drift:\n{manifest_summary_df.loc[drift >= PREVALENCE_TOLERANCE]}"
assert (manifest_summary_df["negative_records"] >= MIN_NEGATIVE_RECORDS).all(), \
    "degenerate experiment with too few negatives"

print(f"\nManifest build runtime: {time.perf_counter() - manifest_started:.1f}s")
print(f"Total sampled records: {manifest_summary_df['sampled_records'].sum():,}")
print("\nPASS: prevalence preserved and no degenerate experiments.\n")

display(
    manifest_summary_df
    .groupby(["dataset", "anomaly_rate"], as_index=False)
    .agg(experiments=("experiment_id", "count"),
         records=("sampled_records", "sum"),
         mean_realized_prevalence=("realized_prevalence", "mean"),
         min_negatives=("negative_records", "min"))
)
memory_report("after manifest")


## 8c. Deterministic rule evaluation

Rules are evaluated on every corrupted experiment and then restricted to the corresponding manifest sample. Duplicate keys are evaluated on the full experiment before sampling so that a sampled subset cannot hide a duplicate pair.

Coverage assertions require the rule engine to fire on the rule-expressible anomaly classes A1, A2, and A3 in all three domains.


In [ ]:
# ======================================================================
# Deterministic rule evaluation, restricted to the manifest.
#
# Duplicate detection runs on the FULL experiment before subsetting, so
# sampling cannot split a duplicate pair and hide it.
#
# Memory: only rule-relevant columns are read. Results are written per
# full-experiment frames — roughly 25 million rows — which is what
# exhausted RAM.
# ======================================================================

import gc
import time

import numpy as np
import pandas as pd


RULE_COLUMNS = {
    "finance":    ["_record_id", "age", "balance", "campaign", "job", "marital",
                   "education", "day", "month"],
    "healthcare": ["_record_id", "encounter_id", "patient_nbr", "time_in_hospital",
                   "number_inpatient"],
    "retail":     ["_record_id", "invoice", "stockcode", "quantity", "price",
                   "invoicedate", "customer_id"],
}

if PURGE_STALE_ARTIFACTS:
    purged = 0
    for path in RULE_BY_EXP_DIR.glob("*.parquet"):
        path.unlink()
        purged += 1
    print(f"Purged {purged} stale rule result files.")

rule_started = time.perf_counter()
rule_coverage_rows = []

for position, experiment in experiment_registry_df.iterrows():
    experiment_id = experiment["experiment_id"]
    dataset       = experiment["dataset"]

    record_path    = RULE_BY_EXP_DIR / f"{experiment_id}_rule_records.parquet"
    violation_path = RULE_BY_EXP_DIR / f"{experiment_id}_rule_violations.parquet"

    if not record_path.exists():
        manifest = pd.read_parquet(
            MANIFEST_DIR / f"{experiment_id}_manifest.parquet",
            columns=["record_id"],
        )
        keep = set(manifest["record_id"].astype(str))

        experiment_df = pd.read_parquet(
            EXPERIMENT_DIR / f"{experiment_id}.parquet",
            columns=RULE_COLUMNS[dataset],
        )

        # Duplicate keys are evaluated on the FULL experiment so that a pair
        # split across the sample boundary is still detected. Only the id set
        # is retained.
        duplicate_keys = [c for c in DUPLICATE_KEYS[dataset]
                          if c in experiment_df.columns]
        duplicate_ids = set(
            experiment_df.loc[
                experiment_df.duplicated(subset=duplicate_keys, keep=False),
                "_record_id",
            ].astype(str)
        ) if duplicate_keys else set()

        # Everything else runs on the 10k sample only. Evaluating the full
        # frame would allocate one Python list per row — 1.17 million of them
        # for retail — which is the single largest avoidable allocation here.
        sample_df = experiment_df.loc[
            experiment_df["_record_id"].astype(str).isin(keep)
        ].copy()
        del experiment_df
        gc.collect()

        results, violations = evaluate_rules(
            sample_df, dataset, duplicate_record_ids=duplicate_ids
        )
        del sample_df, duplicate_ids
        gc.collect()

        results["experiment_id"] = experiment_id
        results = results.rename(columns={"_record_id": "record_id"})
        results["record_id"] = results["record_id"].astype(str)

        # The list-valued column is expensive in parquet and is not needed
        # downstream; violations carry the per-rule detail.
        results = results.drop(columns=["failed_rules"], errors="ignore")
        results.to_parquet(record_path, index=False)

        if not violations.empty:
            violations = violations.loc[
                violations["_record_id"].astype(str).isin(keep)
            ].copy()
            violations["experiment_id"] = experiment_id
            violations = violations.rename(columns={"_record_id": "record_id"})
            violations["record_id"] = violations["record_id"].astype(str)
            if not violations.empty:
                violations.to_parquet(violation_path, index=False)

        del results, violations, manifest, keep
        gc.collect()

    status = pd.read_parquet(record_path, columns=["rule_status"])
    rule_coverage_rows.append({
        "experiment_id": experiment_id,
        "dataset": dataset,
        "anomaly_type": experiment["anomaly_type"],
        "records": len(status),
        "rule_failures": int(status["rule_status"].eq("FAIL").sum()),
    })
    del status
    gc.collect()

    if (position + 1) % 10 == 0:
        memory_report(f"rules {position + 1}/{len(experiment_registry_df)}")

rule_coverage_df = pd.DataFrame(rule_coverage_rows)
rule_coverage_df.to_csv(RESULTS_DIR / "rule_coverage_by_experiment.csv", index=False)

print(f"\nRule evaluation runtime: {time.perf_counter() - rule_started:.1f}s")

coverage_pivot = rule_coverage_df.pivot_table(
    index="dataset", columns="anomaly_type", values="rule_failures", aggfunc="sum"
)
print("\nRule firings by dataset and anomaly class:")
display(coverage_pivot)

# ----------------------------------------------------------------
# The rule engine must detect the classes it is designed for.
# A1 = missingness, A2 = duplicates, A3 = domain violations.
# ----------------------------------------------------------------
for detectable_class in ["A1", "A2", "A3"]:
    fired = rule_coverage_df.loc[
        rule_coverage_df["anomaly_type"].eq(detectable_class), "rule_failures"
    ]
    zero_domains = rule_coverage_df.loc[
        rule_coverage_df["anomaly_type"].eq(detectable_class)
        & rule_coverage_df["rule_failures"].eq(0), "dataset"
    ].unique()
    assert len(zero_domains) == 0, (
        f"Rule engine produced no firings for class {detectable_class} in "
        f"{list(zero_domains)}. Rules must detect A1, A2 and A3 by design."
    )

print("\nPASS: rule engine fires on A1, A2 and A3 in all three domains.")
memory_report("after rules")


## 9. AI-only baseline — Isolation Forest and Local Outlier Factor

Detectors are fitted once per dataset on a held-out clean reference partition and scored on each corrupted experiment. Records used for fitting are excluded from scoring. High-cardinality categorical fields and technical columns are dropped to bound memory.


In [ ]:
# ======================================================================
# Detector fitting: once per dataset, on the held-out clean partition.
#
# Uses the same get_training_record_ids() as the manifest, so records used
# for fitting are provably excluded from scoring.
# ======================================================================

import gc
import json
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler


MAX_CATEGORICAL_CARDINALITY = 100


def select_model_columns(df: pd.DataFrame):
    candidates  = [c for c in df.columns if not c.startswith("_")]
    numeric     = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]
    categorical = [c for c in candidates
                   if c not in numeric
                   and df[c].nunique(dropna=True) <= MAX_CATEGORICAL_CARDINALITY]
    return numeric, categorical


def build_preprocessor(numeric_columns, categorical_columns):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_columns),
        ("categorical", categorical_pipe, categorical_columns),
    ], remainder="drop")


model_bundle = {}
hyperparameter_log = {}

for dataset in ["finance", "healthcare", "retail"]:
    started = time.perf_counter()

    reference_df = pd.read_parquet(BRONZE_DIR / f"bronze_{dataset}.parquet")
    numeric_cols, categorical_cols = select_model_columns(reference_df)
    feature_cols = numeric_cols + categorical_cols

    training_ids = get_training_record_ids(dataset)
    train_mask   = reference_df["_record_id"].astype(str).isin(training_ids)
    train_df     = reference_df.loc[train_mask, feature_cols]

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    X_train = preprocessor.fit_transform(train_df)

    iforest = IsolationForest(
        n_estimators=200, contamination="auto",
        random_state=RANDOM_SEED, n_jobs=-1,
    ).fit(X_train)

    n_neighbors = min(35, max(5, len(train_df) - 1))
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors, novelty=True,
        contamination="auto", n_jobs=-1,
    ).fit(X_train)

    hyperparameters = {
        "dataset": dataset,
        "train_fraction": TRAIN_FRACTION,
        "training_rows": int(len(train_df)),
        "transformed_features": int(X_train.shape[1]),
        "random_seed": RANDOM_SEED,
        "isolation_forest": {"n_estimators": 200, "contamination": "auto",
                             "max_samples": "auto", "random_state": RANDOM_SEED},
        "local_outlier_factor": {"n_neighbors": int(n_neighbors), "novelty": True,
                                 "contamination": "auto", "metric": "minkowski"},
        "numeric_imputer": "median",
        "numeric_scaler": "RobustScaler",
        "categorical_imputer": "most_frequent",
        "categorical_encoder": "OneHotEncoder(handle_unknown=ignore)",
        "max_categorical_cardinality": MAX_CATEGORICAL_CARDINALITY,
        "numeric_columns": len(numeric_cols),
        "categorical_columns": len(categorical_cols),
        "fitting_scope": "once per dataset on a held-out clean reference partition",
    }
    hyperparameter_log[dataset] = hyperparameters

    bundle = {
        "preprocessor": preprocessor,
        "iforest": iforest,
        "lof": lof,
        "numeric_columns": numeric_cols,
        "categorical_columns": categorical_cols,
        "training_record_ids": training_ids,
        "hyperparameters": hyperparameters,
    }
    model_bundle[dataset] = bundle
    joblib.dump(bundle, MODEL_DIR / f"{dataset}_baseline_models.joblib")

    print(f"{dataset:<12} trained on {len(train_df):>7,} rows · "
          f"{X_train.shape[1]:>5,} features · {time.perf_counter() - started:5.1f}s")

    del reference_df, train_df, X_train, train_mask
    gc.collect()

with open(RESULTS_DIR / "detector_hyperparameters.json", "w") as handle:
    json.dump(hyperparameter_log, handle, indent=2)

print("\nHyperparameters written to results/detector_hyperparameters.json")
memory_report("after model fitting")


In [ ]:
# ======================================================================
# Detector scoring, restricted to the manifest.
#
# Existing detector outputs are removed before scoring when
# PURGE_STALE_ARTIFACTS is enabled, ensuring that all artifacts come from
# the current execution.
#
# Output schema is LONG (one row per record per detector), which is what
# Notebook 2 expects.
# ======================================================================

import gc
import time

import numpy as np
import pandas as pd


if PURGE_STALE_ARTIFACTS:
    purged = 0
    for path in AI_RESULT_DIR.glob("*_ai_results.parquet"):
        path.unlink()
        purged += 1
    print(f"Purged {purged} stale AI result files.")
    if purged:
        print("Existing detector artifacts were removed before regeneration.")


def score_experiment(experiment_row: pd.Series) -> pd.DataFrame:
    dataset       = experiment_row["dataset"]
    experiment_id = experiment_row["experiment_id"]
    bundle        = model_bundle[dataset]

    manifest = pd.read_parquet(MANIFEST_DIR / f"{experiment_id}_manifest.parquet")
    wanted   = set(manifest["record_id"].astype(str))

    feature_cols = bundle["numeric_columns"] + bundle["categorical_columns"]
    df = pd.read_parquet(
        EXPERIMENT_DIR / f"{experiment_id}.parquet",
        columns=feature_cols + ["_record_id"],
    )
    df = df.loc[df["_record_id"].astype(str).isin(wanted)].copy()

    # Guard against a manifest/experiment mismatch.
    assert len(df) == len(manifest), (
        f"{experiment_id}: manifest has {len(manifest)} records but "
        f"{len(df)} were found in the experiment file"
    )

    X = bundle["preprocessor"].transform(df[feature_cols])
    record_ids = df["_record_id"].astype(str).values
    del df
    gc.collect()

    frames = []
    for detector_name, model in [("ISOLATION_FOREST", bundle["iforest"]),
                                 ("LOCAL_OUTLIER_FACTOR", bundle["lof"])]:
        started = time.perf_counter()
        scores      = -model.decision_function(X)
        predictions = (model.predict(X) == -1).astype(int)
        runtime     = time.perf_counter() - started

        frames.append(pd.DataFrame({
            "experiment_id": experiment_id,
            "dataset": dataset,
            "anomaly_type": experiment_row["anomaly_type"],
            "anomaly_rate": float(experiment_row["anomaly_rate"]),
            "record_id": record_ids,
            "detector": detector_name,
            "anomaly_score": scores.astype(float),
            "anomaly_prediction": predictions,
            "records_scored": len(record_ids),
            "batch_runtime_seconds": runtime,
        }))

    result = pd.concat(frames, ignore_index=True)
    del X, frames
    gc.collect()
    return result


scoring_started = time.perf_counter()
completed_rows, failed_rows = [], []

print(f"\nScoring {len(experiment_registry_df)} experiments "
      f"({SAMPLE_RECORDS_PER_EXPERIMENT:,} records each)")
print("-" * 78)

for position, (_, experiment_row) in enumerate(experiment_registry_df.iterrows(), start=1):
    experiment_id = experiment_row["experiment_id"]
    output_path   = AI_RESULT_DIR / f"{experiment_id}_ai_results.parquet"
    started       = time.perf_counter()

    if output_path.exists():
        completed_rows.append({"experiment_id": experiment_id,
                               "dataset": experiment_row["dataset"],
                               "status": "SKIPPED_EXISTING",
                               "rows_written": len(pd.read_parquet(
                                   output_path, columns=["record_id"])),
                               "runtime_seconds": 0.0})
        continue

    try:
        result = score_experiment(experiment_row)
        result.to_parquet(output_path, index=False)
        completed_rows.append({
            "experiment_id": experiment_id,
            "dataset": experiment_row["dataset"],
            "anomaly_type": experiment_row["anomaly_type"],
            "anomaly_rate": experiment_row["anomaly_rate"],
            "rows_written": len(result),
            "records_scored": int(result["records_scored"].iloc[0]),
            "status": "COMPLETED",
            "runtime_seconds": time.perf_counter() - started,
        })
        print(f"[{position:>2}/{len(experiment_registry_df)}] {experiment_id:<28} "
              f"rows={len(result):>7,}  {time.perf_counter() - started:6.1f}s")
        del result
    except Exception as exc:
        failed_rows.append({"experiment_id": experiment_id,
                            "error_type": type(exc).__name__,
                            "error_message": str(exc)})
        print(f"[{position:>2}/{len(experiment_registry_df)}] FAILED {experiment_id}: "
              f"{type(exc).__name__}: {exc}")

    gc.collect()
    if position % 10 == 0:
        memory_report(f"scoring {position}/{len(experiment_registry_df)}")

completed_df = pd.DataFrame(completed_rows)
failed_df    = pd.DataFrame(failed_rows)
completed_df.to_csv(RESULTS_DIR / "ai_scoring_completed.csv", index=False)
failed_df.to_csv(RESULTS_DIR / "ai_scoring_failed.csv", index=False)

print("-" * 78)
print(f"Completed: {len(completed_df)} · Failed: {len(failed_df)} · "
      f"Total {(time.perf_counter() - scoring_started) / 60:.1f} min")

# ----------------------------------------------------------------
# Hard validation. Failures abort rather than print.
# ----------------------------------------------------------------
assert failed_df.empty, f"{len(failed_df)} experiments failed:\n{failed_df}"
assert len(completed_df) == len(experiment_registry_df), "missing experiments"

stale = completed_df.loc[completed_df["status"].eq("SKIPPED_EXISTING")]
assert stale.empty, (
    f"{len(stale)} experiments were skipped rather than scored. Set "
    f"PURGE_STALE_ARTIFACTS = True and rerun the scoring stage."
)

expected_rows = SAMPLE_RECORDS_PER_EXPERIMENT * 2
assert (completed_df["rows_written"] == expected_rows).all(), (
    f"expected {expected_rows} rows per experiment, got "
    f"{sorted(completed_df['rows_written'].unique())}"
)

print("\nPASS: all experiments freshly scored at the expected size.")
memory_report("after scoring")


## 10. AI-only evaluation
Metrics are calculated against the explicit injected-anomaly ground truth. Macro summaries across anomaly types and datasets should be used in the paper rather than relying on a single experiment.

In [ ]:
# ======================================================================
# Detector evaluation against manifest ground truth.
#
# Metrics are computed one experiment at a time and appended to a small
# frame. No global concat of record-level data.
# ======================================================================

import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, confusion_matrix


evaluation_started = time.perf_counter()
metric_rows = []

for position, experiment in experiment_registry_df.iterrows():
    experiment_id = experiment["experiment_id"]

    scores_df = pd.read_parquet(AI_RESULT_DIR / f"{experiment_id}_ai_results.parquet")
    truth_df  = pd.read_parquet(
        MANIFEST_DIR / f"{experiment_id}_manifest.parquet",
        columns=["record_id", "ground_truth_label"],
    )

    merged = scores_df.merge(truth_df, on="record_id", how="left", validate="many_to_one")
    assert merged["ground_truth_label"].notna().all(), \
        f"{experiment_id}: unlabelled records after manifest join"

    for detector, group in merged.groupby("detector"):
        y_true = group["ground_truth_label"].to_numpy(dtype=int)
        y_pred = group["anomaly_prediction"].to_numpy(dtype=int)
        y_score = group["anomaly_score"].to_numpy(dtype=float)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        # No experiment may be degenerate at this point.
        assert (tn + fp) > 0, f"{experiment_id}/{detector}: no negative records"

        precision = tp / max(tp + fp, 1)
        recall    = tp / max(tp + fn, 1)
        fpr       = fp / max(fp + tn, 1)

        metric_rows.append({
            "experiment_id": experiment_id,
            "dataset": experiment["dataset"],
            "anomaly_type": experiment["anomaly_type"],
            "anomaly_rate": experiment["anomaly_rate"],
            "detector": detector,
            "true_positive": int(tp), "false_positive": int(fp),
            "false_negative": int(fn), "true_negative": int(tn),
            "precision": precision,
            "recall": recall,
            "f1": 2 * precision * recall / max(precision + recall, 1e-12),
            "false_positive_rate": fpr,
            "false_negative_rate": fn / max(fn + tp, 1),
            "specificity": tn / max(tn + fp, 1),
            "balanced_accuracy": (recall + tn / max(tn + fp, 1)) / 2,
            "pr_auc": average_precision_score(y_true, y_score) if y_true.sum() else np.nan,
            "scored_rows": len(group),
            "realized_prevalence": float(y_true.mean()),
            "runtime_seconds": float(group["batch_runtime_seconds"].iloc[0]),
        })

    del scores_df, truth_df, merged
    gc.collect()

    if (position + 1) % 20 == 0:
        memory_report(f"evaluation {position + 1}/{len(experiment_registry_df)}")

ai_metrics_df = pd.DataFrame(metric_rows)
ai_metrics_df.to_csv(RESULTS_DIR / "ai_evaluation_metrics.csv", index=False)

macro_summary = ai_metrics_df.groupby(["dataset", "anomaly_type", "detector"],
                                      as_index=False).agg(
    macro_precision=("precision", "mean"),
    macro_recall=("recall", "mean"),
    macro_f1=("f1", "mean"),
    mean_fpr=("false_positive_rate", "mean"),
    mean_fnr=("false_negative_rate", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_pr_auc=("pr_auc", "mean"),
    mean_runtime_seconds=("runtime_seconds", "mean"),
)
macro_summary.to_csv(RESULTS_DIR / "ai_macro_summary.csv", index=False)

print(f"Evaluation runtime: {time.perf_counter() - evaluation_started:.1f}s")

# ----------------------------------------------------------------
# Balanced accuracy must reconcile with recall and FPR. A mismatch
# indicates degenerate folds inside the macro-average — this check
# ----------------------------------------------------------------
reconstructed = (ai_metrics_df["recall"] + (1 - ai_metrics_df["false_positive_rate"])) / 2
assert np.allclose(reconstructed, ai_metrics_df["balanced_accuracy"], atol=1e-9), \
    "balanced accuracy does not reconcile with recall and FPR"

print("PASS: balanced accuracy reconciles in every experiment.\n")

# ----------------------------------------------------------------
# Flag trivially separable anomaly classes. Perfect recall with
# PR-AUC at 1.0 means the injection is a giveaway, not that the
# detector is strong. A referee will raise this.
# ----------------------------------------------------------------
trivial = ai_metrics_df.loc[
    (ai_metrics_df["recall"] >= 0.999) & (ai_metrics_df["pr_auc"] >= 0.999)
]
if not trivial.empty:
    print(f"WARNING: {len(trivial)} detector/experiment pairs are trivially separable "
          f"(recall = 1.0 and PR-AUC = 1.0).")
    print("Affected classes:",
          sorted(trivial["anomaly_type"].unique()),
          "| domains:", sorted(trivial["dataset"].unique()))
    print("Constant sentinel injections such as -999 and fixed 12-sigma shifts are")
    print("perfectly detectable. Consider drawing invalid values from a plausible")
    print("range instead, and report per-class results so this is visible.\n")

display(macro_summary.sort_values(["dataset", "anomaly_type", "detector"]))
memory_report("after evaluation")


## 11. Baseline completion checks

The notebook validates experiment counts, realized prevalence, class balance, rule coverage, detector outputs, and required artifact paths before the downstream decision engine is run.


In [ ]:
# ======================================================================
# Baseline completion checks.
#
# Realized prevalence is checked directly against the nominal injection
# rate, and every experiment must retain sufficient positive and negative
# observations for valid metric computation.
# ======================================================================

import gc

import numpy as np
import pandas as pd


manifest_summary_df = pd.read_csv(RESULTS_DIR / "sample_manifest_summary.csv")
rule_coverage_df    = pd.read_csv(RESULTS_DIR / "rule_coverage_by_experiment.csv")
ai_metrics_df       = pd.read_csv(RESULTS_DIR / "ai_evaluation_metrics.csv")
scoring_log_df      = pd.read_csv(RESULTS_DIR / "ai_scoring_completed.csv")

prevalence_drift = (
    ai_metrics_df["realized_prevalence"] - ai_metrics_df["anomaly_rate"]
).abs()

balanced_ok = np.allclose(
    (ai_metrics_df["recall"] + (1 - ai_metrics_df["false_positive_rate"])) / 2,
    ai_metrics_df["balanced_accuracy"], atol=1e-9,
)

rule_fires = rule_coverage_df.groupby("anomaly_type")["rule_failures"].sum()

checks = {
    "three_datasets_loaded":
        len(list(BRONZE_DIR.glob("bronze_*.parquet"))) == 3,
    "all_63_experiments_created":
        len(experiment_registry_df) == 63,
    "manifest_for_every_experiment":
        len(manifest_summary_df) == 63,
    "realized_prevalence_matches_nominal":
        bool((prevalence_drift < PREVALENCE_TOLERANCE).all()),
    "no_degenerate_experiments":
        bool((ai_metrics_df["true_negative"] + ai_metrics_df["false_positive"]
              >= MIN_NEGATIVE_RECORDS).all()),
    "balanced_accuracy_reconciles": bool(balanced_ok),
    "rules_run_per_experiment":
        rule_coverage_df["experiment_id"].nunique() == 63,
    "rules_detect_A1": int(rule_fires.get("A1", 0)) > 0,
    "rules_detect_A2": int(rule_fires.get("A2", 0)) > 0,
    "rules_detect_A3": int(rule_fires.get("A3", 0)) > 0,
    "no_stale_ai_artifacts":
        not scoring_log_df["status"].eq("SKIPPED_EXISTING").any(),
    "two_detectors_per_experiment":
        len(ai_metrics_df) == 63 * 2,
    "hyperparameters_exported":
        (RESULTS_DIR / "detector_hyperparameters.json").exists(),
}

checks_df = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
display(checks_df)

failed_checks = checks_df.loc[~checks_df["passed"], "check"].tolist()
assert not failed_checks, f"Baseline checks failed: {failed_checks}"

print("\nAll baseline integrity checks passed.")
print(f"Records per experiment : {SAMPLE_RECORDS_PER_EXPERIMENT:,}")
print(f"Total scored records   : {ai_metrics_df['scored_rows'].sum() // 2:,}")
print(f"Mean realized prevalence drift: {prevalence_drift.mean():.6f}")
print("\nNotebook 1 complete. Proceed to 02_Hybrid_Decision_Engine_MEMSAFE.ipynb.")
memory_report("final")


## 12. Downstream analysis

The companion decision-engine notebook consumes these artifacts to perform evidence fusion, governed routing, nested calibration, statistical analysis, operational-burden evaluation, and publication-artifact generation.


## 13. Completion Summary

This cell reports the principal artifact locations created by the baseline pipeline. The generated files are consumed by the hybrid decision notebook.

In [ ]:
from pathlib import Path

# Ensure path variables are defined in case of session reset
if 'PROJECT_ROOT' not in locals():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Paper2_ZTLF")

# Re-derive standard subdirectories
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"

artifact_locations = {
    "project_root": PROJECT_ROOT,
    "bronze_data": BRONZE_DIR,
    "silver_data": SILVER_DIR,
    "experiment_data": EXPERIMENT_DIR,
    "models": MODEL_DIR,
    "results": RESULTS_DIR,
}

# Ensure experiment_registry_df is available for the print statement
if 'experiment_registry_df' not in locals():
    import pandas as pd
    registry_path = RESULTS_DIR / 'experiment_registry.csv'
    experiment_registry_df = pd.read_csv(registry_path) if registry_path.exists() else pd.DataFrame()

print(f"Configured experiments: {len(experiment_registry_df):,}")
for name, path in artifact_locations.items():
    print(f"{name}: {path}")